In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
import json

In [2]:
df = pd.read_csv("../data/processed/train_scaled.csv")

print(df.shape)
df.head()

(20631, 23)


,engine_id,cycle,op_setting_1,op_setting_2,op_setting_3,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,...,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_20,sensor_21,RUL
0,1,1,-0.0007,-0.0004,100.0,-1.721725,-0.134255,-0.925936,-5.329071e-15,0.141683,...,-0.266467,0.334262,-1.058890,-0.269071,-0.603816,-3.469447e-18,-0.781710,1.348493,1.194427,125
1,1,2,0.0019,-0.0003,100.0,-1.061780,0.211528,-0.643726,-5.329071e-15,0.141683,...,-0.191583,1.174899,-0.363646,-0.642845,-0.275852,-3.469447e-18,-0.781710,1.016528,1.236922,125
2,1,3,-0.0043,0.0003,100.0,-0.661813,-0.413166,-0.525953,-5.329071e-15,0.141683,...,-1.015303,1.364721,-0.919841,-0.551629,-0.649144,-3.469447e-18,-2.073094,0.739891,0.503423,125
3,1,4,0.0007,0.0000,100.0,-0.661813,-1.261314,-0.784831,-5.329071e-15,0.141683,...,-1.539489,1.961302,-0.224597,-0.520176,-1.971665,-3.469447e-18,-0.781710,0.352598,0.777792,125
4,1,5,-0.0019,-0.0002,100.0,-0.621816,-1.251528,-0.301518,-5.329071e-15,0.141683,...,-0.977861,1.052871,-0.780793,-0.521748,-0.339845,-3.469447e-18,-0.136018,0.463253,1.059552,125


In [3]:
with open("../data/processed/top_sensors.json") as f:
    sensors = json.load(f)

print(sensors)

['sensor_11', 'sensor_9', 'sensor_4', 'sensor_12', 'sensor_7', 'sensor_14', 'sensor_15', 'sensor_21', 'sensor_13', 'sensor_2']


In [4]:
## no leakage
engine_ids = df["engine_id"].unique()

train_engines, test_engines = train_test_split(
    engine_ids,
    test_size=0.2,
    random_state=42
)

train_df = df[df["engine_id"].isin(train_engines)]
test_df = df[df["engine_id"].isin(test_engines)]

print("Train engines:", len(train_engines))
print("Test engines:", len(test_engines))

Train engines: 80
Test engines: 20


In [5]:
def create_windows(df, sensors, window):

    X = []
    y = []

    for engine in df["engine_id"].unique():

        engine_df = df[df["engine_id"] == engine].sort_values("cycle")

        sensor_vals = engine_df[sensors].values
        rul_vals = engine_df["RUL"].values

        for i in range(len(engine_df) - window + 1):

            window_data = sensor_vals[i:i+window]
            target = rul_vals[i+window-1]

            X.append(window_data.flatten())
            y.append(target)

    return np.array(X), np.array(y)

In [6]:
window_sizes = [5,10,20,30,40]

output_dir = Path("../data/processed/windows_no_leak")
output_dir.mkdir(parents=True, exist_ok=True)

for w in window_sizes:

    print(f"\nWindow size {w}")

    X_train, y_train = create_windows(train_df, sensors, w)
    X_test, y_test = create_windows(test_df, sensors, w)

    np.savez(
        output_dir / f"window_{w}.npz",
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test
    )

    print("Train shape:", X_train.shape)
    print("Test shape:", X_test.shape)


Window size 5
Train shape: (16241, 50)
Test shape: (3990, 50)

Window size 10
Train shape: (15841, 100)
Test shape: (3890, 100)

Window size 20
Train shape: (15041, 200)
Test shape: (3690, 200)

Window size 30
Train shape: (14241, 300)
Test shape: (3490, 300)

Window size 40
Train shape: (13441, 400)
Test shape: (3290, 400)
